# node总结
前面langgraph的持久化与记忆管理就讲完了，最后总结node节点

总结完整的结构，和使用的底层逻辑

## 节点的本质
1. 常用方式，写def同步函数
比较复杂的话，也可以写async 异步节点，  同步的话如果有一些磁盘写入，会进入线程等待，不会有资源的释放，异步的话可以释放资源，效率更高

如果涉及节点有大量的磁盘写入，可以使用异步方式，根据python基础的同步或者异步来选择，同个graph也可以部分同步，部分异步

2. 除了写函数，也可以是Runnable实例，和函数差别不大，  一般在创建新的线程的时候，用到Runnable语法， 一般写函数就行
3. 从源码看， 节点最终会变成特殊的可运行对象，对象叫做`PregelNode.bound`， bound就是确定保存下来，后面能看到这个对象


## 节点函数的完整形态
> 节点是一个函数， python函数涉及 输入参数和输出结果
完整形态

所有能够使用的参数
1. state 状态， 最常用， 有不同类型 私有， 输入， 全局
2. config 运行图的配置， 可以读取config["configurable"]["thread_id"], config["recursion_limit"], config["metadata"]， 
    - 图是大的框架，  节点是某个执行的工人， 工人可以到旁边的位置领取工具， 工具就是config这些，需要查看当前图的相关配置
3. runtime: `def llm_node(state: OverAllState, runtime: Runtime[UserContext]) -> OverAllState:` 只对当前有效的上下文参数 ， 也可以获取长期记忆的store
4. writer：流式写入器，后面课程有，但是可以理解为，如果不适用，也可以完成graph的使用，相当于一个高级的功能

```python
def node(state: State, config: RunnableConfig, runtime: Runtime[Context]) -> State:
    ...
```

没有流式写入器，最长上面的

* 编译图时传入 **`checkpointer`**，并在调用时传入 **`config`**，可以在节点中通过 **`config`** 间接访问当前调用的配置信息，如 **`thread_id`**。
* 编译图时传入 **`store`** 后，可以在节点中通过 **`runtime.store`** 访问长期记忆存储器。
* 初始化状态图时传入 **`context_schema`**，并在调用时传入 **`context`** 后，可以在节点中通过 **`runtime.context`** 访问运行时上下文。

## 节点的触发
1. 用户视角，通过边，来指定触发，一种最简单的普通边，固定决定后续的节点
2. 条件边，
    - 专门添加路由函数，等路由函数执行完后， 根据返回的内容确定，走的是那个节点
    - 也可以返回Send实例，动态派发多个并行任务，常用于`Map-Reduce`场景
3. 奇葩场景 Command.goto 动态触发后面的节点， 在node执行里， 不在edge里

---

1. 源码视角：将图状态和节点之间的触发关系组织为通道，有多种通道
    - state的状态通道
    - 节点之间的控制流关系，叫做触发通道，或者屏障通道

**例子1**

例如，对于普通边：

```python
builder.add_edge("node_a", "node_b")
```

运行时 **`node_a`** 的写入器会向以下通道**写入数据：**

```text
branch:to:node_b
```

而 **`node_b`** 会将该通道注册为自己的**触发通道。**


**例子2**  
条件边和 **`Command.goto`** 最终也会根据目标节点生成相应的触发通道写入。**`Send`** 则会生成动态派发任务所需的 **`Send`** 数据包。

对于多个前驱节点共同汇聚到同一个节点的情况，底层还可能使用类似下面的屏障通道：

```text
join:<node_a>+<node_b>:<node_c>
```

用于等待指定的前驱节点全部完成。

当某个节点订阅的触发通道在当前超步中产生新的写入后，该节点会在下一超步被调度。

## 节点的执行
用户层面，六个步骤
1. 获取node的四个入参， 哪些需要使用的参数，去graph里面读取， 准备好输入参数
2. 执行node的业务逻辑
3. 返回state的更新， 或者Command跳转等内容
4. Langgraph将上一步node返回的state要更新的值，转换为状态通道和控制通道的写入条目
5. 如果当前节点挂载了条件边，则执行条件边路由逻辑
6. 最后 上面去拿不完成了， langgraph把所有内容汇总， 比如state调用reducer全部合并图状态


注意，节点不需要返回全部字段，只需要返回要更新的，  因为状态在底层会通过通道方式合并

---
源码层面  

会有pegalnode构建， pegalnode里核心的是bound.runnable是节点函数，  writers

其中：

* **`bound`** 保存节点的业务逻辑；
* **`writers`** 保存节点执行完成后需要调用的写入器；
* **`triggers`** 保存能够触发该节点的通道。

可以近似表示为：

```text
PregelNode
├── bound：节点业务逻辑
├── writers：状态和控制流写入器
└── triggers：能够触发该节点的通道
```

因此，从整体上看，节点执行并不是简单地”调用一个函数并返回结果”，而是：

> 执行业务逻辑，生成状态和控制流写入，并通过目标节点触发通道或动态任务写入，为下一超步生成待调度任务，持续推动计算图运行。 （比较官方翻译）